# M0 · Start: dane TechRetail w Twoim workspace

Jesteś analitykiem w **TechRetail Corp**, dystrybutorze elektroniki dla firm B2B. Zarząd chce asystenta, który odpowiada na pytania o klientów. Część odpowiedzi jest w tabeli, a część w raportach PDF analityków. Przez cały dzień budujemy tego asystenta: od rozmowy w AI Playground do agenta, który sam wybiera między SQL a dokumentami.

Ten notebook przygotowuje wszystko, czego potrzebują moduły M1-M6 we wszystkich trzech ścieżkach: **A · Razem** (TechRetail, robimy wspólnie), **B · Samodzielnie** (sieć piekarni Bakehouse) i **C · Wyzwanie**.

| Krok | Co powstaje |
|---|---|
| 1 | tabela `workspace.default.gold_customer_360`: 28 813 wierszy (28 670 klientów), 19 kolumn |
| 2 | Volume `retail_docs` z 10 raportami PDF |
| 3 | tabela `retail_rag_chunks` z gotowymi fragmentami raportów |
| 3b | dane ścieżek B i C: kopie Bakehouse w `workspace.bakehouse` (transakcje, franczyzy, opinie) i oferty Airbnb w `workspace.airbnb.listings` |
| 4 | start endpointu AI Search `retail_rag_search` (w tle, gotowy na M3) |
| 5 | preflight: SQL, model, tracing i dane ścieżek B i C, tabela "działa / nie działa" |

**Środowisko:** Databricks Free Edition, Serverless. Uruchom notebook przyciskiem **Run all** i czytaj dalej, zanim skończy.

> Dane są syntetyczne i spseudonimizowane: `customer_name` = `Customer <id>`, `tax_id` ma fikcyjne wartości. Pochodzą z datasetu Databricks Marketplace, który prowadzący przygotował wcześniej.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

**Infrastruktura.** Konfiguracja wspólna dla wszystkich modułów. Uruchom i czytaj dalej, tu nie ma nic do nauczenia.


In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

**Infrastruktura.** Katalog i cztery schematy warsztatu: `default` (ścieżka A, TechRetail), `bakehouse` i `airbnb` (ścieżki B i C) oraz `governance` na maski i filtry. Na Free Edition `workspace.default` istnieje od początku. W workspace firmowym albo trialu katalog bywa utworzony bez schematu `default`, więc komórka zakłada to, czego brakuje. Uruchom i czytaj dalej.

In [ ]:
# Unity Catalog nazywa dane trójstopniowo: katalog → schemat → tabela. Tu zakładamy brakujące poziomy.
# Katalog utworzony przez API albo Terraform nie dostaje schematu `default`, więc nie zakładamy, że jest.
catalogs = {row[0] for row in spark.sql("SHOW CATALOGS").collect()}
if CATALOG not in catalogs:
    # Gdy to polecenie skończy się błędem uprawnień: poproś administratora o katalog `workspace`
    # z prawami USE CATALOG, USE SCHEMA i CREATE SCHEMA. Na Free Edition ten katalog jest od początku.
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

# default: ścieżka A (TechRetail); bakehouse i airbnb: ścieżki B i C; governance: ich maski i filtry.
for schema in (SCHEMA, BH_SCHEMA, AIRBNB_SCHEMA, POLICY_SCHEMA):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

# USE ustawia domyślny katalog i schemat, żeby dalej pisać samą nazwę tabeli zamiast pełnej ścieżki.
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Katalog {CATALOG} gotowy, schematy: {SCHEMA}, {BH_SCHEMA}, {AIRBNB_SCHEMA}, {POLICY_SCHEMA}")


**Infrastruktura.** Ścieżki do danych w repozytorium. Uruchom i czytaj dalej.


In [ ]:
import os
from pathlib import Path

# Notebook leży w workshop/00_setup/, dane w workshop/data/ tego samego folderu Git.
DATA_DIR = Path(os.getcwd()).parent / "data"
USERNAME = spark.sql("SELECT current_user()").first()[0]

missing = [p for p in ("tables/gold_customer_360.parquet", "checkpoints/retail_rag_chunks.parquet", "documents") if not (DATA_DIR / p).exists()]
assert not missing, f"Brak plików w {DATA_DIR}: {missing}. Zaimportuj repozytorium jako folder Git (README, prework)."
print(f"Użytkownik: {USERNAME}\nDane: {DATA_DIR}")

## 1. Tabela Gold: `gold_customer_360`

> **Cel:** mieć w swoim katalogu tabelę, na której pracuje cały dzień.
> **Gotowe, gdy:** `gold_customer_360` istnieje i ma **28 813 wierszy**.


Jeden wiersz to jeden klient B2B z cechami RFM, z jednym wyjątkiem: **143 klientów ma po dwa identyczne wiersze**, więc 28 813 wierszy to 28 670 klientów. Stąd przez cały dzień rozróżniamy "liczbę wierszy" i "liczbę klientów" (w M4 zobaczysz 9 541 kontra 9 494). Cechy RFM: **R**ecency (dni od ostatniego zakupu), **F**requency (liczba pozycji), **M**onetary (wartość zakupów). Do tego segment lojalności 0-3 i lokalizacja. Tabelę wczytujesz z gotowego pliku.

Zapis jako tabela Delta w Unity Catalog daje transakcje ACID, wersjonowanie (Time Travel) i uprawnienia, z których skorzystamy w M4.

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

gold_pdf = pd.read_parquet(DATA_DIR / "tables" / "gold_customer_360.parquet")
gold_df = (
    spark.createDataFrame(gold_pdf)
    .withColumn("first_order_date", F.col("first_order_date").cast("date"))
    .withColumn("last_order_date", F.col("last_order_date").cast("date"))
)

# Po przerwanym M4 (zadanie z maską) na tabeli mogą zostać row filter i maska: zdejmujemy je przed zapisem.
if spark.catalog.tableExists(GOLD_TABLE):
    for statement in (f"ALTER TABLE {GOLD_TABLE} DROP ROW FILTER",
                      f"ALTER TABLE {GOLD_TABLE} ALTER COLUMN tax_id DROP MASK"):
        try:
            spark.sql(statement)
        except Exception:
            pass  # polityki nie było: nic do zdjęcia

(gold_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)
print(f"Tabela zapisana: {GOLD_TABLE}")
print(f"  Wiersze: {spark.table(GOLD_TABLE).count():,} (oczekiwane: 28,813)")
print(f"  Kolumny: {len(spark.table(GOLD_TABLE).columns)} (oczekiwane: 19)")

In [ ]:
%sql
-- Segmenty lojalności: 0=Nowi/Nieaktywni, 1=Rozwijający się, 2=Regularni, 3=VIP
SELECT
  loyalty_segment,
  COUNT(*)                        AS klienci,
  ROUND(AVG(monetary), 2)         AS avg_monetary,
  ROUND(AVG(recency_days), 0)     AS avg_recency_days,
  SUM(CASE WHEN num_orders = 0 THEN 1 ELSE 0 END) AS bez_zamowien
FROM workspace.default.gold_customer_360
GROUP BY loyalty_segment
ORDER BY loyalty_segment

In [ ]:
%sql
-- Geografia: gdzie są klienci?
SELECT state, COUNT(*) AS klienci, ROUND(AVG(monetary), 2) AS avg_monetary
FROM workspace.default.gold_customer_360
GROUP BY state
ORDER BY klienci DESC
LIMIT 10

In [ ]:
# ai_query(): wywołanie modelu bezpośrednio w SQL — 4 wiersze = 4 wywołania.
display(spark.sql(f"""
WITH segment_stats AS (
  SELECT loyalty_segment, COUNT(*) AS customers, ROUND(AVG(monetary), 2) AS avg_monetary
  FROM {GOLD_TABLE}
  GROUP BY loyalty_segment
)
SELECT
  loyalty_segment,
  customers,
  avg_monetary,
  ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
      'Jesteś analitykiem retail. Segment lojalności ', CAST(loyalty_segment AS STRING),
      ' ma ', CAST(customers AS STRING), ' klientów, średnia wartość zakupów: ', CAST(avg_monetary AS STRING), ' USD. ',
      'Segment 0=nowy, 1=okazjonalny, 2=regularny, 3=VIP. ',
      'Podaj JEDNĄ krótką rekomendację biznesową (max 20 słów) po polsku.'
    )
  ) AS ai_recommendation
FROM segment_stats
ORDER BY loyalty_segment
"""))

In [ ]:
%sql
-- Time Travel: każdy zapis tworzy nową wersję tabeli
DESCRIBE HISTORY workspace.default.gold_customer_360

## 2. Raporty PDF w Volume

> **Cel:** mieć raporty analityków tam, skąd sięgnie po nie RAG w M3.
> **Gotowe, gdy:** w Volume `retail_docs` leży **10 plików PDF**.


Analitycy TechRetail przygotowali 10 raportów o segmentach, geografii, retencji, wartości klientów, churnie i jakości danych. Wrzucamy je do **Volume**, czyli przestrzeni na pliki zarządzanej przez Unity Catalog. W M3 zamienią się w wiedzę, z której korzysta asystent.

In [ ]:
import shutil

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Serverless nie obsługuje dbutils.fs.cp z file: — kopiujemy przez system plików Volume.
pdfs = sorted((DATA_DIR / "documents").glob("*.pdf"))
for pdf in pdfs:
    shutil.copy(pdf, Path(VOLUME_PATH) / pdf.name)

print(f"Pliki w {VOLUME_PATH}:")
for f in sorted(dbutils.fs.ls(VOLUME_PATH), key=lambda x: x.name):
    if f.name.endswith(".pdf"):
        print(f"   • {f.name} ({f.size / 1024:,.0f} KB)")
assert len(pdfs) == 10, f"Oczekiwano 10 PDF, jest {len(pdfs)}"

## 3. Fragmenty raportów: `retail_rag_chunks`

> **Cel:** zobaczyć, że dokument trafia do wyszukiwarki pocięty na fragmenty, a nie w całości.
> **Gotowe, gdy:** tabela `retail_rag_chunks` ma fragmenty z nazwą raportu i numerem.


Model nie czyta całych PDF. Wyszukiwarka zwraca mu kilka krótkich **chunków**, czyli fragmentów po ok. 600 znaków. W M3 zobaczysz, jak powstają (`ai_parse_document` → tekst → podział). Tu wczytujesz gotowy wynik, żeby indeks AI Search mógł powstać od razu.

Tabela potrzebuje **klucza głównego** i **Change Data Feed**. Dzięki temu indeks synchronizuje tylko zmienione wiersze.

In [ ]:
chunks_pdf = pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_chunks.parquet")
spark.createDataFrame(chunks_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CHUNKS_TABLE)

spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} ALTER COLUMN chunk_id SET NOT NULL")
try:
    spark.sql(f"ALTER TABLE {CHUNKS_TABLE} ADD CONSTRAINT pk_chunk_id PRIMARY KEY (chunk_id)")
except Exception as e:
    if "already exists" not in str(e).lower():
        raise

n_chunks = spark.table(CHUNKS_TABLE).count()
print(f"{CHUNKS_TABLE}: {n_chunks} chunków z 10 raportów (PK chunk_id, CDF włączony)")
display(spark.table(CHUNKS_TABLE).groupBy("doc_id").count().orderBy("doc_id"))

## 3b. Dane ścieżek B i C: Bakehouse i Airbnb

> **Cel:** mieć dane do ścieżek B i C, każde we własnym schemacie.
> **Gotowe, gdy:** `workspace.airbnb.listings` ma **8 533** oferty, a w `workspace.bakehouse` są trzy tabele.


Ścieżka A pracuje na TechRetail w schemacie `default`. Ścieżka B (sieć piekarni Bakehouse) dostaje kopie `samples.bakehouse` w schemacie `bakehouse`, a zadania C na Airbnb pracują na `workspace.airbnb.listings`. Osobne schematy mają konkretny powód: w M6 serwer MCP wystawia agentowi **wszystkie** funkcje ze schematu, więc funkcje jednej ścieżki nie mogą trafić do agenta drugiej.

Oferty Airbnb w San Francisco leżą w repozytorium. Imiona gospodarzy i współrzędne zostały usunięte już w pliku (licencja CC BY 4.0, Inside Airbnb, patrz `data/NOTICE.md`).

In [ ]:
# Ścieżka C: oferty Airbnb we własnym schemacie. Typy i flagi jakości ustawiamy od razu,
# żeby Genie i funkcje nie liczyły średniej z ceny 0 USD ani nie sortowały dat jak tekstu.
airbnb = pd.read_csv(DATA_DIR / "practice" / "sf_airbnb_listings.csv")
airbnb["price"] = pd.to_numeric(airbnb["price"], errors="coerce")
airbnb = airbnb.drop(columns=["host_name", "host_id", "latitude", "longitude", "neighbourhood_group"], errors="ignore").dropna(subset=["id", "price"])
airbnb["last_review"] = pd.to_datetime(airbnb["last_review"], format="%d-%m-%Y", errors="coerce").dt.date
airbnb["price_valid"] = airbnb["price"].between(1, 2000)          # 2 oferty po 0 USD i 27 powyżej 2000 USD
airbnb["is_short_term"] = airbnb["minimum_nights"] < 30           # 45% ofert to najem od 30 nocy
airbnb["name"] = airbnb["name"].fillna("")
airbnb = airbnb.astype({"last_review": "object"}).where(airbnb.notna(), None)
spark.createDataFrame(airbnb).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(AIRBNB_TABLE)
listings = spark.table(AIRBNB_TABLE)
print(f"{AIRBNB_TABLE}: {listings.count():,} ofert, dzielnic: {listings.select('neighbourhood').distinct().count()}, "
      f"krótkoterminowych: {listings.where('is_short_term').count():,}")

**Infrastruktura (ścieżka B).** Kopie trzech tabel `samples.bakehouse` w schemacie `bakehouse`. Katalog `samples` jest tylko do odczytu, a na kopii da się nałożyć maskę i zbudować wyszukiwanie po opiniach z kluczem `review_id`. Uruchom i czytaj dalej.

In [ ]:
try:
    spark.sql(f"CREATE OR REPLACE TABLE {BH_TRANSACTIONS} AS SELECT * FROM samples.bakehouse.sales_transactions")
    spark.sql(f"CREATE OR REPLACE TABLE {CATALOG}.{BH_SCHEMA}.franchises AS SELECT * FROM samples.bakehouse.sales_franchises")
    spark.sql(f"""CREATE OR REPLACE TABLE {BH_REVIEWS}
        TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
        SELECT sha2(concat_ws('|', CAST(franchiseID AS STRING), CAST(review_date AS STRING), review), 256) AS review_id,
               franchiseID, review_date, review
        FROM samples.bakehouse.media_customer_reviews
        WHERE length(review) > 20""")
    for table in (BH_TRANSACTIONS, f"{CATALOG}.{BH_SCHEMA}.franchises", BH_REVIEWS):
        print(f"{table}: {spark.table(table).count():,} wierszy")
except Exception as e:
    print(f"❌ Nie udało się skopiować samples.bakehouse: {type(e).__name__}: {str(e)[:200]}")
    print("   Katalog samples jest w tym workspace niedostępny albo nie masz do niego dostępu. "
          "Ścieżka B i capstone Bakehouse nie zadziałają, ścieżki A i C tak. Zgłoś to prowadzącemu; notebook idzie dalej.")


## 4. Endpoint AI Search (dawniej Vector Search)

> **Cel:** wystartować endpoint wyszukiwarki **teraz**, żeby był gotowy po lunchu.
> **Gotowe, gdy:** endpoint `retail_rag_search` istnieje; stan `PROVISIONING` jest w porządku.


**AI Search** to zarządzana wyszukiwarka wektorowa i pełnotekstowa Databricks. Do połowy 2026 roku nazywała się Vector Search. Pierwsze uruchomienie endpointu trwa od kilku do kilkunastu minut, dlatego startujemy go teraz i **nie czekamy**. Do M3 będzie gotowy.

Na Free Edition masz **jeden endpoint**. Jeśli już istnieje, ta komórka go nie rusza.

In [ ]:
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)
try:
    if search_client.endpoint_exists(SEARCH_ENDPOINT):
        state = search_client.get_endpoint(SEARCH_ENDPOINT).get("endpoint_status", {}).get("state")
        print(f"Endpoint {SEARCH_ENDPOINT} już istnieje (stan: {state})")
    else:
        search_client.create_endpoint(name=SEARCH_ENDPOINT, endpoint_type="STANDARD")
        print(f"Uruchamiam endpoint {SEARCH_ENDPOINT} w tle — stan pokaże preflight poniżej")
except Exception as e:
    print(f"Nie udało się uruchomić endpointu: {type(e).__name__}: {e}")
    print("Nic straconego: M3 ma tryb offline (retrieve_local) na przygotowanych embeddingach.")

## 5. Preflight

> **Cel:** wiedzieć **przed** pierwszym modułem, co u Ciebie działa, a co nie.
> **Gotowe, gdy:** w tabeli wyników masz ✅ przy SQL, modelu i tracingu.


Sprawdzamy rzeczy, na których opiera się reszta dnia: dane TechRetail, model, tracing, AI Search oraz dane ścieżek B (Bakehouse) i C (Airbnb), z których korzysta też capstone. Jeśli któraś jest na czerwono, zgłoś to prowadzącemu **teraz**, a nie w M3.

**Jak działa preflight.** Komórka niżej przygotowuje pustą listę wyników i jedną funkcję, która wykona każde sprawdzenie. Sama jeszcze niczego nie sprawdza.

- `check(name, fn)` przyjmuje nazwę sprawdzenia i funkcję do uruchomienia. Uruchamia ją i dopisuje do listy `results` trzy rzeczy: nazwę, czy się udało, i wynik. Gdy funkcja rzuci błąd, `check()` go łapie, zapisuje jego rodzaj i początek komunikatu, i idzie dalej. Dzięki temu jedno czerwone sprawdzenie nie zatrzymuje pozostałych.


In [ ]:
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
results = []


def check(name, fn):
    try:
        results.append((name, True, fn()))
    except Exception as e:
        results.append((name, False, f"{type(e).__name__}: {str(e)[:160]}"))

**Siedem sprawdzeń.** Każde jest niezależne: błąd jednego nie przerywa reszty, bo `check()` łapie wyjątek i zapisuje go jako ❌ w tabeli. Dzięki temu w jednym przebiegu widzisz **pełny** obraz środowiska, a nie pierwszy napotkany problem.

Proste sprawdzenia (liczba wierszy w tabelach TechRetail, Bakehouse i Airbnb) są wpisane w miejscu jako `lambda`. Trzy wymagają kilku linii i mają własne funkcje:

- `call_llm()` wysyła do modelu z konfiguracji jedno krótkie pytanie z naszym `SYSTEM_PROMPT` i zwraca nazwę endpointu razem z odpowiedzią. Jeśli tu jest błąd, w M1 nie odpowie żaden model.
- `trace()` ustawia Twój eksperyment MLflow w folderze `/Users/<Twój login>/` i zapisuje w nim jeden testowy ślad. Zwraca ścieżkę eksperymentu. To ten sam eksperyment, w którym w kolejnych modułach zobaczysz ślady agenta.
- `search_state()` pyta o stan endpointu AI Search i zwraca go jako tekst. `PROVISIONING` jest w porządku, endpoint dopiero się uruchamia. Gotowy będzie potrzebny w M3.

Na końcu komórka wypisuje tabelę: znak, nazwa sprawdzenia, wynik. Czerwone pozycje zgłoś prowadzącemu.


In [ ]:
check("SQL na tabeli Gold", lambda: f"{spark.table(GOLD_TABLE).where('loyalty_segment = 3').count():,} wierszy VIP".replace(",", " ") + " (oczekiwane 9 541 wierszy = 9 494 klientów, bo 143 klientów ma po dwa wiersze)")
check("Chunki raportów", lambda: f"{spark.table(CHUNKS_TABLE).count()} wierszy")


def call_llm():
    client = w.serving_endpoints.get_open_ai_client()
    reply = client.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Odpowiedz jednym słowem: gotowy?"}],
        max_tokens=10,
    )
    return f"{LLM_ENDPOINT}: {reply.choices[0].message.content.strip()!r}"


check("Model (Foundation Model API)", call_llm)


def trace():
    mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
    with mlflow.start_span(name="preflight") as span:
        span.set_inputs({"check": "tracing"})
        span.set_outputs({"ok": True})
    return f"eksperyment /Users/{USERNAME}/{EXPERIMENT_NAME}"


check("MLflow Tracing", trace)


def search_state():
    from databricks.ai_search.client import AISearchClient
    client = AISearchClient(disable_notice=True)
    return client.get_endpoint(SEARCH_ENDPOINT).get("endpoint_status", {}).get("state", "?")


check("Endpoint AI Search (może być PROVISIONING)", search_state)
check("Ścieżka B: Bakehouse (schemat bakehouse)",
      lambda: (f"{spark.table(BH_TRANSACTIONS).count():,} transakcji, {spark.table(BH_REVIEWS).count():,} opinii").replace(",", " "))
check("Ścieżka C: Airbnb (schemat airbnb)", lambda: f"{spark.table(AIRBNB_TABLE).count():,} ofert".replace(",", " ") + " (oczekiwane 8 533)")

print(f"{'':2} {'Sprawdzenie':<44} Wynik")
for name, ok, detail in results:
    print(f"{'✅' if ok else '❌'} {name:<44} {detail}")

## Lab: poznaj dane, zanim zapytasz o nie model

> **Cel:** poznać dane, zanim zaczniesz o nie pytać model.
> **Gotowe, gdy:** potrafisz powiedzieć, ile jest klientów VIP i co oznacza `loyalty_segment`.


**Na koniec:** otwórz `workshop/transfer/canvas_agenta.md` i wpisz domenę, dla której chcesz zbudować agenta pod koniec dnia, oraz 5 pytań jej użytkowników. Jeśli nie masz własnej domeny, wpisz "sieć piekarni Bakehouse".

Odpowiedz na trzy pytania **samym SQL**. Dodaj komórkę poniżej i zapisz odpowiedzi.

1. Ilu jest klientów VIP (`loyalty_segment = 3`) i jaka jest ich średnia `monetary`? Podaj obie wartości: po wierszach (`AVG(monetary)` → 1038,72 USD) i po klientach (`DISTINCT customer_id` → 1043,15 USD, tę zwróci narzędzie w M2).
2. Który stan ma najwięcej klientów?
3. Jaki odsetek klientów nie złożył żadnego zamówienia (`num_orders = 0`)?

<details><summary>Oczekiwane wyniki (sprawdź po swojej próbie)</summary>

1. 9 541 wierszy VIP (9 494 klientów), średnia `monetary` ≈ 1038,72 USD
2. NY, 3 417 klientów
3. 26 862 z 28 813 wierszy, czyli ok. 93%. Większość bazy to klienci bez historii zamówień w oknie danych.

</details>

## Podsumowanie

- `gold_customer_360` to **dane ustrukturyzowane**: liczby, na które odpowiada SQL.
- `retail_docs` i `retail_rag_chunks` to **wiedza nieustrukturyzowana**: wnioski i rekomendacje, których w tabeli nie ma.
- Asystent, którego budujemy, musi umieć skorzystać z obu źródeł i wiedzieć, kiedy żadne nie pasuje.

**Dalej:** `demo/m1_agentic_ai_playground` (prowadzący) i `labs/m1_agentic_ai_playground` (Ty).